# Homework Option C: Flower Prompt Gallery and Curation

Time: up to 60 minutes

This notebook reuses the pretrained CLIP-conditioned flowers model from Part 3. There is very little new code here. The focus is on designing creative prompts and writing a short curator's report on what worked.
Look for `# TODO` comments to add your own content.
See `Homework_OptionC_FlowerGallery.md` for full instructions.

Requires: Google Drive with the pretrained flowers model from Part 3, same as the workshop.

In [ ]:
!pip install -q git+https://github.com/openai/CLIP.git

In [ ]:
import torch
import clip
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

clip_model, clip_preprocess = clip.load('ViT-B/32', device=device)
clip_model.eval()
CLIP_DIM = 512
print('CLIP loaded.')

---

## Setup: Load the Pretrained Flowers Model

Same model and noise schedule as the Part 3 demo. Already working.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

FLOWER_CH, FLOWER_SIZE = 3, 32
T_FLOWERS = 300

B_f = torch.linspace(0.0001, 0.02, T_FLOWERS).to(device)
a_f = 1.0 - B_f
a_bar_f = torch.cumprod(a_f, dim=0)
sqrt_a_bar_f = torch.sqrt(a_bar_f)
sqrt_one_minus_a_bar_f = torch.sqrt(1 - a_bar_f)
sqrt_a_inv_f = torch.sqrt(1 / a_f)
pred_noise_coeff_f = (1 - a_f) / sqrt_one_minus_a_bar_f

@torch.no_grad()
def reverse_q_f(x_t, t, e_t):
    t_int = t.int()
    u_t = sqrt_a_inv_f[t_int,None,None,None] * (x_t - pred_noise_coeff_f[t_int,None,None,None] * e_t)
    if t_int[0] == 0: return u_t
    return u_t + torch.sqrt(B_f[t_int-1,None,None,None]) * torch.randn_like(x_t)

# UNetCFG class, same architecture as Part 3 (img_ch=3, img_size=32, c_embed_dim=512)
# NOTE: copy the UNetCFG class definition from your Part 3 notebook here before running.

FLOWERS_MODEL_PATH = '/content/drive/MyDrive/BestCourse2026/workshop-genai/flowers_clip.pth'

model_flowers = UNetCFG(T_FLOWERS, img_ch=FLOWER_CH, img_size=FLOWER_SIZE,
                        down_chs=(32,64,128), c_embed_dim=CLIP_DIM).to(device)
model_flowers.load_state_dict(torch.load(FLOWERS_MODEL_PATH, map_location=device))
model_flowers.eval()
print('Pretrained flowers model loaded.')

---

## Step 1: Design Your Prompts

Write 5 to 6 of your own creative text prompts. Be more adventurous than the workshop demo: unusual colors, moods, or combinations. They do not all have to work, that is part of the experiment.

In [ ]:
# TODO: write 5 to 6 of your own creative text prompts
my_prompts = [
    '',  # TODO: prompt 1
    '',  # TODO: prompt 2
    '',  # TODO: prompt 3
    '',  # TODO: prompt 4
    '',  # TODO: prompt 5
]

<details>
<summary><b>Click to show example prompts if you are stuck</b></summary>

```python
my_prompts = [
    'A black rose with silver edges',
    'A daisy made of flames',
    'A pastel rainbow sunflower',
    'A rose covered in morning dew',
    'A sunflower at midnight under the moon',
]
```

These are just inspiration. Your own ideas are just as valid, even if some of them fail.
</details>

---

## Step 2: Generate the Gallery

This generates one image per prompt at a fixed guidance weight. Already working, runs once `my_prompts` is filled in.

In [ ]:
@torch.no_grad()
def generate_gallery(prompts, w=1.5):
    tokens = clip.tokenize(prompts).to(device)
    c_clip = clip_model.encode_text(tokens).float()
    c_clip = c_clip / c_clip.norm(dim=-1, keepdim=True)

    n = len(prompts)
    w_t = torch.full((n, 1, 1, 1), w, device=device)

    x_t = torch.randn(n, FLOWER_CH, FLOWER_SIZE, FLOWER_SIZE, device=device)
    for i in range(T_FLOWERS - 1, -1, -1):
        t = torch.full((n,), i, device=device).float()
        x2 = x_t.repeat(2,1,1,1); t2 = t.repeat(2)
        c2 = c_clip.repeat(2, 1)
        m2 = torch.cat([torch.ones_like(c_clip), torch.zeros_like(c_clip)], dim=0)
        e2 = model_flowers(x2, t2, c2, m2)
        e_t = (1 + w_t) * e2[:n] - w_t * e2[n:]
        x_t = reverse_q_f(x_t, t, e_t)
    return x_t

gallery = generate_gallery(my_prompts, w=1.5)

fig, axes = plt.subplots(1, len(my_prompts), figsize=(3*len(my_prompts), 3))
for ax, img, prompt in zip(axes, gallery, my_prompts):
    ax.imshow(((img+1)/2).clamp(0,1).permute(1,2,0).cpu())
    ax.set_title(prompt[:22], fontsize=8)
    ax.axis('off')
plt.suptitle('Your prompt gallery (w=1.5)')
plt.tight_layout()
plt.show()

---

## Step 3: Compare Guidance Weights

Pick 2 of your prompts (the ones you are most curious about) and compare them at a weak and a strong guidance weight.

In [ ]:
# TODO: pick 2 of your prompts to compare
comparison_prompts = [
    '',  # TODO: prompt A
    '',  # TODO: prompt B
]
comparison_weights = [0.5, 2.0]

fig, axes = plt.subplots(len(comparison_weights), len(comparison_prompts),
                         figsize=(3*len(comparison_prompts), 3*len(comparison_weights)))
for row, w_val in enumerate(comparison_weights):
    imgs = generate_gallery(comparison_prompts, w=w_val)
    for col, (img, prompt) in enumerate(zip(imgs, comparison_prompts)):
        ax = axes[row, col]
        ax.imshow(((img+1)/2).clamp(0,1).permute(1,2,0).cpu())
        if row == 0: ax.set_title(prompt[:22], fontsize=8)
        if col == 0: ax.set_ylabel(f'w={w_val}', fontsize=9)
        ax.axis('off')
plt.suptitle('Weak vs strong guidance')
plt.tight_layout()
plt.show()

---

## Your Report (Curator's Notes)

Answer these questions in this cell:

**1. Look at your Step 2 gallery. Which prompt produced the most convincing flower, and which one failed the most? What do you think caused the difference?**

_(your answer here)_

**2. Look at your Step 3 comparison. How did the result change between the weak and strong guidance weight, for each of your two prompts?**

_(your answer here)_

**3. If you wrote one more prompt to fix the failed one from question 1, what would you change about the wording?**

_(your answer here)_